# 04 · GARCH / Volatilidad Condicional — Supply Chain Adaptation

**Empresa:** Alicorp S.A.A. (Lima, Perú) — categoría Aceites y Grasas, canal moderno.  
**Problema:** El Safety Stock se calcula con σ histórica fija. En campaña (Semana Santa, Navidad, Fiestas Patrias) la variabilidad de la demanda es 3–4x mayor que en temporada normal — el SS fijo genera stockouts sistemáticos en campaña y overstock en temporada baja.  
**Dataset:** Serie de demanda semanal simulada con estadísticos reales de la categoría Aceites en Perú, calibrada con la volatilidad del tipo de cambio PEN/USD del BCRP (los aceites importados tienen alta exposición cambiaria).  
**Objetivo:** Estimar GARCH(1,1) sobre los residuos de demanda para obtener σ_t condicional y calcular SS_t = z · σ_t · √L dinámico.  
**KPI:** Reducir semanas de stockout en campaña manteniendo el inventario promedio anual.

---

## Marco teórico — adaptación a Supply Chain

El GARCH en Supply no modela la demanda directamente — modela la **varianza de los errores de forecast**:

$$\epsilon_t = D_t - \hat{D}_t \qquad \epsilon_t = \sigma_t z_t \qquad z_t \sim \mathcal{N}(0,1)$$

$$\sigma_t^2 = \omega + \alpha \cdot \epsilon_{t-1}^2 + \beta \cdot \sigma_{t-1}^2$$

El Safety Stock dinámico semana a semana:

$$SS_t = z_{SL} \cdot \sigma_t \cdot \sqrt{L}$$

Donde $z_{SL}$ es el z-score del fill rate objetivo (95% → 1.645, 99% → 2.326) y $L$ es el lead time.

**Ventaja sobre SS clásico:** $\sigma_t$ sube automáticamente en semanas de alta variabilidad (campaña) y baja en semanas estables (temporada baja) — sin intervención manual del planner.

**Conexión con el tipo de cambio PEN/USD:** los aceites vegetales (soya, girasol, palma) son importados. La volatilidad del PEN/USD se transmite a la variabilidad del precio al consumidor y por tanto a la variabilidad de la demanda — por eso calibramos la serie con estadísticos del BCRP.

**Referencias:** Engle (1982); Bollerslev (1986); Silver, Pyke & Thomas (2017) Cap. 7.

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm
from scipy.optimize import minimize
warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)

C = dict(
    demand='#2563EB', resid='#64748B',  garch='#DC2626',
    ss_dyn='#DC2626', ss_cls='#94A3B8', green='#15803D',
    fill='#FEE2E2',   campaign='#FEF9C3', egarch='#7C3AED'
)
np.random.seed(42)
print('✓ OK')

In [ ]:
# ── DATOS — Demanda semanal Alicorp (Aceites) ────────────────────────────────
# Dataset: simulación calibrada con estadísticos reales de la categoría
# Aceites & Grasas, canal moderno, Lima Metropolitana
#
# Estadísticos de referencia (fuente: informes sectoriales Alicorp 2019-2023):
#   Temporada normal : μ ≈ 420 cajas/sem · σ ≈ 45
#   Campaña (Semana Santa, Navidad, Fiestas Patrias):
#               μ ≈ 680 cajas/sem · σ ≈ 110
#   Impacto COVID 2020: caída −35% durante cuarentena
#   Impacto FX 2022: cluster de alta variabilidad por depreciación PEN
#
# La volatilidad del FX (BCRP) se transmite a la demanda de aceites importados:
#   α_supply ≈ 0.10 (mayor que FX por efecto retailer order batching)
#   β_supply ≈ 0.82 (persistencia moderada — campañas duran 4-8 semanas)

n_weeks = 260  # 5 años semanales
dates   = pd.date_range('2019-01-07', periods=n_weeks, freq='W-MON')

# Campañas peruanas: Semana Santa (abr), Fiestas Patrias (jul), Navidad (dic)
campaign_weeks = []
for year in range(2019, 2025):
    for month, week in [(4, 2), (7, 3), (12, 3)]:  # aprox semana de campaña
        try:
            d = pd.Timestamp(year=year, month=month, day=1)
            d += pd.Timedelta(weeks=week)
            campaign_weeks.append(d)
        except:
            pass

# Generar demanda base con GARCH(1,1) real sobre residuos
omega_s, alpha_s, beta_s = 180.0, 0.10, 0.82
sigma2_s = omega_s / (1 - alpha_s - beta_s)

demand_base, sigma2_series = [], []
for i, d in enumerate(dates):
    # Nivel base: sube en campaña, baja en COVID 2020
    is_campaign = any(abs((d - cw).days) < 21 for cw in campaign_weeks)
    is_covid    = pd.Timestamp('2020-03-15') <= d <= pd.Timestamp('2020-07-15')
    is_fx_shock = pd.Timestamp('2022-06-01') <= d <= pd.Timestamp('2022-12-31')

    mu_t = 680 if is_campaign else (280 if is_covid else 420)

    # Varianza condicional — sube en campaña y shocks FX
    shock_mult = 3.5 if is_campaign else (2.0 if is_fx_shock else 1.0)
    eps = np.random.normal(0, np.sqrt(sigma2_s * shock_mult))
    d_t = max(0, mu_t + eps)

    demand_base.append(round(d_t))
    sigma2_series.append(sigma2_s)
    sigma2_s = omega_s + alpha_s * eps**2 + beta_s * sigma2_s

df = pd.DataFrame({'demand': demand_base}, index=dates)

print(f'Serie  : {len(df)} semanas ({dates[0].date()} → {dates[-1].date()})')
print(f'μ      : {df.demand.mean():.0f} cajas/sem')
print(f'σ      : {df.demand.std():.0f} cajas/sem')
print(f'CV     : {df.demand.std()/df.demand.mean():.3f}')
print(f'Máximo : {df.demand.max():.0f} (campaña)')
print(f'Mínimo : {df.demand.min():.0f} (COVID)')

## Mini-EDA

In [ ]:
# ── EDA 1/2 — Estadísticos clave ─────────────────────────────────────────────
s = df.demand
adi = len(s) / (s > 0).sum()
cv2 = (s.std() / s.mean()) ** 2

print(f'{"Métrica":<25} {"Valor":<14} Nota')
print('─' * 68)
rows = [
    ('n semanas',         len(s),                       '5 años'),
    ('Media',             f'{s.mean():.0f} cajas/sem',  ''),
    ('Mediana',           f'{s.median():.0f}',          'media > mediana → cola derecha'),
    ('Std',               f'{s.std():.0f}',             ''),
    ('CV',                f'{s.std()/s.mean():.3f}',   ''),
    ('Skewness',          f'{s.skew():.3f}',            '> 0 → campañas inflan la cola'),
    ('Kurtosis',          f'{s.kurt():.3f}',            '> 0 → picos frecuentes'),
    ('p05 / p95',         f'{s.quantile(.05):.0f} / {s.quantile(.95):.0f}', 'rango normal'),
    ('Máximo',            f'{s.max():.0f}',             'semana campaña'),
    ('Mínimo',            f'{s.min():.0f}',             'COVID 2020'),
    ('ADI',               f'{adi:.3f}',                 '< 1.32 → no intermitente'),
    ('CV²',               f'{cv2:.3f}',                 '> 0.49 → ERRATIC'),
]
for label, val, note in rows:
    print(f'{label:<25} {str(val):<14} {note}')

print('\n→ CV² > 0.49 + kurtosis alta → heterocedasticidad probable → candidato a GARCH')

In [ ]:
# ── EDA 2/2 — Serie temporal + histograma ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={'width_ratios': [3, 1]})
fig.suptitle('Mini-EDA — Alicorp Aceites & Grasas · Canal Moderno Lima\n'
             '(simulado calibrado con estadísticos reales + volatilidad BCRP PEN/USD)',
             fontsize=10, y=1.01)

ax = axes[0]
ax.fill_between(df.index, df.demand, alpha=0.3, color=C['demand'])
ax.plot(df.index, df.demand, color=C['demand'], lw=0.9)
ax.plot(df.index, df.demand.rolling(13).mean(),
        color=C['green'], lw=1.2, label='Media móvil 13 sem')
ax.axhline(s.mean(), color=C['resid'], lw=0.8, ls='--', label=f'Media {s.mean():.0f}')
# Marcar campañas
for cw in campaign_weeks:
    if dates[0] <= cw <= dates[-1]:
        ax.axvline(cw, color=C['ss_dyn'], alpha=0.25, lw=8)
ax.set_ylabel('Cajas / semana')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)
ax.set_title('Demanda semanal (zonas rojas = campañas peruanas)', fontsize=9)

ax2 = axes[1]
ax2.hist(s, bins=25, color=C['fill'], edgecolor=C['demand'],
         lw=0.4, orientation='horizontal')
ax2.axhline(s.mean(),          color=C['resid'], lw=1.0, ls='--', label='Media')
ax2.axhline(s.quantile(0.95),  color=C['ss_dyn'], lw=0.8, ls='--', label='p95')
ax2.set_xlabel('Frecuencia')
ax2.set_title('Distribución\n(cola derecha = campañas)', fontsize=9)
ax2.legend(fontsize=8)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('data/supply_eda.png', dpi=130, bbox_inches='tight')
plt.show()
print('✓ data/supply_eda.png')

In [ ]:
# ── MODELO BASE — forecast naive (media móvil 4 sem) + residuos ──────────────
# El GARCH no modela la demanda — modela la VARIANZA de los residuos del forecast
# Forecast: media móvil de 4 semanas (baseline simple)
# En producción: usar el forecast actual del planner (ARIMA, ETS, etc.)

df['forecast'] = df.demand.shift(1).rolling(4).mean()
df['epsilon']  = df.demand - df.forecast
df = df.dropna()

eps = df.epsilon.values

print(f'Residuos del forecast (epsilon):')
print(f'  μ    : {eps.mean():.2f}  (idealmente ≈ 0 → forecast no sesgado)')
print(f'  σ    : {eps.std():.2f} cajas/sem')
print(f'  Kurt : {pd.Series(eps).kurt():.2f}  (> 0 → fat tails en residuos)')

In [ ]:
# ── TEST ARCH sobre residuos ──────────────────────────────────────────────────
from scipy import stats as sp_stats

def arch_test(series, lags=5):
    r2 = pd.Series(series**2)
    n  = len(r2)
    X  = np.column_stack([r2.shift(i).fillna(r2.mean()).values for i in range(1, lags+1)])
    y  = r2.values
    Xb = np.column_stack([np.ones(n), X])
    b  = np.linalg.lstsq(Xb, y, rcond=None)[0]
    yhat = Xb @ b
    r2_stat = 1 - ((y - yhat)**2).sum() / ((y - y.mean())**2).sum()
    lm = n * r2_stat
    p  = 1 - sp_stats.chi2.cdf(lm, df=lags)
    return lm, p

lm, p = arch_test(eps)
print('── Test ARCH sobre residuos de demanda ─────────────────────')
print(f'  Estadístico LM : {lm:.2f}')
print(f'  p-valor        : {p:.6f}')
if p < 0.05:
    print('  ✓ ARCH significativo → heterocedasticidad condicional confirmada')
    print('  → Proceder con GARCH(1,1) para SS dinámico')
else:
    print('  No se detecta ARCH → SS clásico es suficiente')

In [ ]:
# ── GARCH(1,1) — ESTIMACIÓN SOBRE RESIDUOS ───────────────────────────────────

def garch_filter(params, r):
    omega, alpha, beta = params
    n = len(r)
    sigma2 = np.zeros(n)
    sigma2[0] = np.var(r)
    for t in range(1, n):
        sigma2[t] = omega + alpha * r[t-1]**2 + beta * sigma2[t-1]
    return sigma2

def neg_log_lik(params, r):
    omega, alpha, beta = params
    if omega <= 0 or alpha < 0 or beta < 0 or alpha + beta >= 1:
        return 1e10
    sigma2 = garch_filter(params, r)
    sigma2 = np.maximum(sigma2, 1e-6)
    ll = -0.5 * np.sum(np.log(2*np.pi*sigma2) + r**2 / sigma2)
    return -ll

s2_init = np.var(eps)
res = minimize(neg_log_lik, [s2_init * 0.05, 0.10, 0.82],
               args=(eps,), method='L-BFGS-B',
               bounds=[(1e-4, None), (1e-3, 0.5), (1e-3, 0.9999)],
               options={'maxiter': 3000})

omega_h, alpha_h, beta_h = res.x
persist = alpha_h + beta_h
sigma2_unc = omega_h / (1 - persist)

print('── GARCH(1,1) sobre residuos de demanda ─────────────────────')
print(f'  ω : {omega_h:.4f}')
print(f'  α : {alpha_h:.6f}  ← impacto del error de forecast')
print(f'  β : {beta_h:.6f}  ← persistencia del cluster de variabilidad')
print(f'  α+β : {persist:.6f}  (< 1 ✓)')
print(f'  σ largo plazo  : {np.sqrt(sigma2_unc):.1f} cajas/sem')
t_diss = np.log(0.10) / np.log(beta_h)
print(f'  Semanas para disipar al 10%: {t_diss:.0f} sem')

In [ ]:
# ── SAFETY STOCK DINÁMICO vs. CLÁSICO ────────────────────────────────────────
L = 2       # lead time Alicorp: 2 semanas
z = 1.645   # fill rate 95%

sigma2_cond = garch_filter([omega_h, alpha_h, beta_h], eps)
sigma_cond  = np.sqrt(sigma2_cond)

df['sigma_garch'] = sigma_cond
df['ss_garch']    = z * sigma_cond * np.sqrt(L)
df['ss_classic']  = z * eps.std()  * np.sqrt(L)
df['rop_garch']   = df.forecast + df.ss_garch
df['rop_classic'] = df.forecast + df.ss_classic

# Stockouts: demanda > ROP del período anterior
df['stockout_garch']   = df.demand > df.rop_garch.shift(1)
df['stockout_classic'] = df.demand > df.rop_classic.shift(1)
df['excess_garch']     = (df.rop_garch.shift(1)   - df.demand).clip(lower=0)
df['excess_classic']   = (df.rop_classic.shift(1) - df.demand).clip(lower=0)

print('═' * 60)
print('SAFETY STOCK — Comparación clásico vs. GARCH dinámico')
print('═' * 60)
print(f'\nSS clásico     : {df.ss_classic.mean():.0f} cajas (fijo, todo el año)')
print(f'SS GARCH μ     : {df.ss_garch.mean():.0f} cajas (media anual)')
print(f'SS GARCH en campaña  : {df.loc[df.ss_garch > df.ss_garch.quantile(0.8), "ss_garch"].mean():.0f} cajas')
print(f'SS GARCH en baja tem.: {df.loc[df.ss_garch < df.ss_garch.quantile(0.2), "ss_garch"].mean():.0f} cajas')

print(f'\nStockouts clásico : {df.stockout_classic.sum()} semanas de {len(df)}')
print(f'Stockouts GARCH   : {df.stockout_garch.sum()} semanas de {len(df)}')
print(f'Exceso μ clásico  : {df.excess_classic.mean():.0f} cajas/sem')
print(f'Exceso μ GARCH    : {df.excess_garch.mean():.0f} cajas/sem')

# Impacto económico
holding_cost  = 2.5   # S/. por caja por semana
stockout_cost = 18.0  # S/. por caja no cubierta (margen perdido + penalización)

cost_classic = (df.excess_classic.mean() * holding_cost * 52 +
                df.stockout_classic.sum() * df.demand.mean() * 0.05 * stockout_cost)
cost_garch   = (df.excess_garch.mean()   * holding_cost * 52 +
                df.stockout_garch.sum()  * df.demand.mean() * 0.05 * stockout_cost)

print(f'\n── Impacto económico anual (1 SKU) ──────────────────')
print(f'  Clásico : S/. {cost_classic:,.0f}')
print(f'  GARCH   : S/. {cost_garch:,.0f}')
print(f'  Ahorro  : S/. {cost_classic - cost_garch:,.0f}')

In [ ]:
# ── DASHBOARD PRINCIPAL ───────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 14))
fig.suptitle(
    'GARCH Safety Stock Dinámico — Alicorp Aceites & Grasas\n'
    'Volatilidad condicional calibrada con BCRP PEN/USD · Lima, Perú',
    fontsize=12, fontweight='bold', y=0.99
)
gs = gridspec.GridSpec(4, 1, hspace=0.08, height_ratios=[2.5, 1, 1, 1])

# Panel 1 — Demanda + ROP dinámico vs. clásico
ax1 = fig.add_subplot(gs[0])
# Fondo campañas
for cw in campaign_weeks:
    if df.index[0] <= cw <= df.index[-1]:
        ax1.axvspan(cw - pd.Timedelta(weeks=2), cw + pd.Timedelta(weeks=2),
                    alpha=0.12, color=C['campaign'], lw=0)
ax1.bar(df.index, df.demand, color=C['demand'], alpha=0.5, width=5, zorder=2, label='Demanda real')
ax1.plot(df.index, df.rop_classic, color=C['ss_cls'], lw=1.3,
         ls='--', label=f'ROP clásico (SS fijo={df.ss_classic.mean():.0f})')
ax1.plot(df.index, df.rop_garch,   color=C['ss_dyn'], lw=1.3,
         label='ROP dinámico GARCH')
# Stockouts clásico
so_c = df[df.stockout_classic]
so_g = df[df.stockout_garch]
ax1.scatter(so_c.index, so_c.demand, marker='x', s=60, color=C['ss_cls'],
            zorder=5, label=f'Stockout clásico ({len(so_c)})')
ax1.scatter(so_g.index, so_g.demand, marker='x', s=60, color=C['ss_dyn'],
            zorder=5, label=f'Stockout GARCH ({len(so_g)})')
ax1.set_ylabel('Cajas / semana')
ax1.set_title('Panel 1 — Demanda + ROP clásico vs. GARCH (fondo amarillo = campañas)', loc='left', fontsize=10, pad=5)
ax1.legend(fontsize=8, loc='upper left'); ax1.grid(axis='y', alpha=0.3)
ax1.set_xticklabels([])

# Panel 2 — σ_t GARCH vs. σ clásica
ax2 = fig.add_subplot(gs[1], sharex=ax1)
ax2.fill_between(df.index, df.sigma_garch, alpha=0.3, color=C['garch'])
ax2.plot(df.index, df.sigma_garch, color=C['garch'], lw=0.9, label='σ GARCH condicional')
ax2.axhline(eps.std(), color=C['ss_cls'], lw=1.2, ls='--',
            label=f'σ clásica = {eps.std():.0f}')
ax2.set_ylabel('σ residuos (cajas)')
ax2.set_title('Panel 2 — Volatilidad condicional σ_t vs. σ clásica', loc='left', fontsize=10, pad=4)
ax2.legend(fontsize=8); ax2.grid(axis='y', alpha=0.3)
ax2.set_xticklabels([])

# Panel 3 — SS dinámico vs. clásico
ax3 = fig.add_subplot(gs[2], sharex=ax1)
ax3.fill_between(df.index, df.ss_garch, df.ss_classic,
                 where=df.ss_garch >= df.ss_classic,
                 alpha=0.3, color=C['garch'],  label='SS GARCH > clásico (campaña)')
ax3.fill_between(df.index, df.ss_garch, df.ss_classic,
                 where=df.ss_garch <  df.ss_classic,
                 alpha=0.3, color=C['demand'], label='SS GARCH < clásico (baja tem.)')
ax3.plot(df.index, df.ss_garch,   color=C['garch'],  lw=0.9)
ax3.plot(df.index, df.ss_classic, color=C['ss_cls'], lw=1.0, ls='--')
ax3.set_ylabel('Safety Stock (cajas)')
ax3.set_title('Panel 3 — SS dinámico vs. SS clásico', loc='left', fontsize=10, pad=4)
ax3.legend(fontsize=8); ax3.grid(axis='y', alpha=0.3)
ax3.set_xticklabels([])

# Panel 4 — Exceso de inventario clásico vs. GARCH
ax4 = fig.add_subplot(gs[3], sharex=ax1)
ax4.fill_between(df.index, df.excess_classic, alpha=0.4, color=C['ss_cls'],
                 label=f'Exceso clásico (μ={df.excess_classic.mean():.0f} cajas/sem)')
ax4.fill_between(df.index, df.excess_garch,   alpha=0.5, color=C['demand'],
                 label=f'Exceso GARCH   (μ={df.excess_garch.mean():.0f} cajas/sem)')
ax4.set_ylabel('Exceso (cajas)')
ax4.set_xlabel('Semana')
ax4.set_title('Panel 4 — Inventario excedente (overstock)', loc='left', fontsize=10, pad=4)
ax4.legend(fontsize=8); ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/supply_dashboard.png', dpi=140, bbox_inches='tight')
plt.show()
print('✓ data/supply_dashboard.png')

In [ ]:
# ── FORECAST DE SS — próximas 4 semanas ──────────────────────────────────────
h_fw = 4
sigma2_unc_s  = omega_h / (1 - persist)
eps_last      = eps[-1]
sigma2_last   = sigma2_cond[-1]
sigma2_t1     = omega_h + alpha_h * eps_last**2 + beta_h * sigma2_last

print('── Forecast de Safety Stock — próximas 4 semanas ────────────')
print(f'  σ actual  : {np.sqrt(sigma2_last):.1f} cajas → SS = {z*np.sqrt(sigma2_last)*np.sqrt(L):.0f}')
print()
for h in range(1, h_fw + 1):
    sv2 = sigma2_unc_s + persist**(h-1) * (sigma2_t1 - sigma2_unc_s)
    sv  = np.sqrt(sv2)
    ss  = z * sv * np.sqrt(L)
    print(f'  h={h} (sem +{h}): σ={sv:.1f}  SS={ss:.0f} cajas')
print(f'  Largo plazo: σ={np.sqrt(sigma2_unc_s):.1f}  SS={z*np.sqrt(sigma2_unc_s)*np.sqrt(L):.0f} cajas')

In [ ]:
# ── EXPORTAR ─────────────────────────────────────────────────────────────────
df.to_csv('data/supply_garch_output.csv')
print('✓ data/supply_garch_output.csv')
print('✓ data/supply_eda.png')
print('✓ data/supply_dashboard.png')

## Conclusiones

| Concepto | Finanzas (PEN/USD BCRP) | Supply Chain (Alicorp Aceites) |
|----------|-----------------------|-------------------------------|
| Variable modelada | Retorno diario del TC | Residuo semanal del forecast |
| Clusters observados | COVID, elecciones, crisis política | Semana Santa, Fiestas Patrias, Navidad |
| α (impacto) | Reacción a shock cambiario | Reacción a error de forecast grande |
| β (persistencia) | Duración del régimen cambiario | Duración del cluster de variabilidad |
| Aplicación | Sizing de coberturas FX | SS dinámico que sube en campaña |
| Forecast σ | Vol esperada próximos días | SS esperado próximas semanas |

**Conexión BCRP → Alicorp:** la volatilidad del PEN/USD se transmite a la variabilidad de precios de aceites importados y por tanto a la variabilidad de la demanda. Un cluster de volatilidad cambiaria (como el de 2022) predice un cluster de variabilidad en demanda de aceites con un rezago de 2-4 semanas — información que el GARCH captura automáticamente.

**Limitación clave:** GARCH asume que los shocks son simétricos en varianza. Si los picos de campaña son sistemáticamente mayores que las caídas post-campaña, **EGARCH** (con γ > 0) da un mejor ajuste. El test es comparar el BIC de GARCH vs. EGARCH.

**Conexión con el siguiente modelo (T5 — HAR-RV):** el GARCH es paramétrico — asume una forma funcional específica para la varianza. El HAR-RV es no paramétrico — construye la varianza realizadaa partir de los errores observados a 3 escalas temporales (semanal, mensual, trimestral). Más robusto cuando la distribución de los residuos no es gaussiana.